# Fraud Model — score the new 100,000 customers\n\nTrains a regressor on the seed 100,000 customers to predict `FraudRiskScore`, then applies it to the new 100,000 and writes into `FactFraudSignal`. `FraudFlag` and `AlertLevel` are derived from the predicted score using the same thresholds as the warehouse generator (Flag if > 0.35; High/Medium/Low bands at 0.35/0.20).\n\nSame caveat as every notebook here: `FraudRiskScore` in the seed data is a formula of `CustomerId`, not real transaction behavior.

In [1]:
from datetime import date

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

from db_utils import bulk_insert, get_connection

FEATURE_COLS = [
    "Age", "Gender", "Region", "CustomerType", "Segment", "CustomerStatus",
    "Balance", "AccountType",
]
TODAY = date.today()

conn = get_connection()
seed = pd.read_sql(
    """
    SELECT c.CustomerId, c.Age, c.Gender, c.Region, c.CustomerType, c.Segment, c.CustomerStatus,
           a.Balance, a.AccountType,
           f.FraudRiskScore
    FROM dbo.DimCustomer c
    JOIN dbo.FactCustomerAccount a ON a.CustomerId = c.CustomerId
    JOIN dbo.FactFraudSignal f ON f.CustomerId = c.CustomerId
    WHERE c.CustomerId <= 100000;
    """,
    conn,
)
conn.close()

new_customers = pd.read_csv("data/new_customers_features.csv")
print("Seed:", seed.shape, " New:", new_customers.shape)

/var/folders/xy/sts9z3rj3w3_957f1n6sltrm0000gn/T/ipykernel_39046/4256322940.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  seed = pd.read_sql(


Seed: (100000, 10)  New: (100000, 15)


In [2]:
combined = pd.concat([seed[FEATURE_COLS], new_customers[FEATURE_COLS]], keys=["seed", "new"])
combined_encoded = pd.get_dummies(combined, drop_first=True)

X_seed = combined_encoded.loc["seed"].reset_index(drop=True)
X_new = combined_encoded.loc["new"].reset_index(drop=True)
y_seed = seed["FraudRiskScore"].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(X_seed, y_seed, test_size=0.2, random_state=42)
eval_model = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
eval_model.fit(X_train, y_train)
y_pred = eval_model.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, y_pred), 4))
print("R2:", round(r2_score(y_test, y_pred), 4))

MAE: 0.0659
R2: 0.6346


In [3]:
final_model = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
final_model.fit(X_seed, y_seed)

predicted_score = np.clip(final_model.predict(X_new), 0, 1).round(4)

results = pd.DataFrame({
    "CustomerId": new_customers["CustomerId"],
    "FraudRiskScore": predicted_score,
    "FraudFlag": (predicted_score > 0.35).astype(int),
    "AlertLevel": np.select(
        [predicted_score > 0.35, predicted_score > 0.20],
        ["High", "Medium"],
        default="Low",
    ),
    "ModelDate": TODAY,
})

print(results["AlertLevel"].value_counts())
results.head()

AlertLevel
Low       45460
Medium    39012
High      15528
Name: count, dtype: int64


,CustomerId,FraudRiskScore,FraudFlag,AlertLevel,ModelDate
0,100001,0.3589,1,High,2026-07-23
1,100002,0.1209,0,Low,2026-07-23
2,100003,0.1411,0,Low,2026-07-23
3,100004,0.3185,0,Medium,2026-07-23
4,100005,0.2785,0,Medium,2026-07-23


In [4]:
cols = ["CustomerId", "FraudRiskScore", "FraudFlag", "AlertLevel", "ModelDate"]

conn = get_connection()
n = bulk_insert(conn, "dbo.FactFraudSignal", cols, list(results[cols].itertuples(index=False, name=None)))
check = pd.read_sql("SELECT COUNT(*) AS NewFraudRows FROM dbo.FactFraudSignal WHERE CustomerId > 100000;", conn)
conn.close()
print(f"Inserted {n:,} rows into FactFraudSignal")
check

Inserted 100,000 rows into FactFraudSignal


/var/folders/xy/sts9z3rj3w3_957f1n6sltrm0000gn/T/ipykernel_39046/2572676463.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  check = pd.read_sql("SELECT COUNT(*) AS NewFraudRows FROM dbo.FactFraudSignal WHERE CustomerId > 100000;", conn)


,NewFraudRows
0,100000
